**NOTE**: This notebook uses default configurations for fine-tuning a model from Hugging Face. You can replace the `MODEL_NAME` variable with a supportive model and full fine-tune it. (i.e. This is using `AutoTokenizer.from_pretrained()`, `AutoModelForSequenceClassification.from_pretrained()`)

In [1]:
# ============================================================
# 01. Import libraries
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
# ============================================================
# 02. Experiment configuration
# ============================================================

MODEL_NAME = "NLPC-UOM/SinBERT-small" 
# ["NLPC-UOM/SinBERT-large", "cardiffnlp/twitter-xlm-roberta-base", etc]

# Change these according to your dataset.
TEXT_COLUMN = "noisy_text"
LABEL_COLUMN = "label"

# Number of classification classes.
NUM_LABELS = 2

MAX_LENGTH = 128

# Checkpoint saving directory
OUTPUT_DIR = ""

# Trained model saving directory
FINAL_MODEL_DIR = ""

# Results saving path (should end with .csv)
RESULTS_PATH = ""

print("Model:", MODEL_NAME)
print("Max sequence length:", MAX_LENGTH)
print("Number of labels:", NUM_LABELS)

Model: NLPC-UOM/SinBERT-large
Max sequence length: 128
Number of labels: 2


In [3]:
# ============================================================
# 03. Set random seeds
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


In [4]:
# ============================================================
# 04. Load JSONL dataset
# ============================================================

TRAIN_PATH = ""
TEST_PATH = ""

TEST_CLEAN_SINHALA = ""
TEST_TRANSLITERATION = ""
TEST_CODE_MIX = ""
TEST_MISSPELL = ""
TEST_COMPOUND = ""

train_df = pd.read_json(TRAIN_PATH, lines=True)
test_df = pd.read_json(TEST_PATH, lines=True)

test_transliteration_df = pd.read_json(TEST_TRANSLITERATION, lines=True)
test_codemix_df = pd.read_json(TEST_CODE_MIX, lines=True)
test_clean_sinhala_df = pd.read_json(TEST_CLEAN_SINHALA, lines=True)
test_misspell_df = pd.read_json(TEST_MISSPELL, lines=True)
test_compound_df = pd.read_json(TEST_COMPOUND, lines=True)

if "clean-sinhala-train-and-test" in TRAIN_PATH: 
    train_df = train_df.rename(columns={"text": TEXT_COLUMN})
    test_df = test_df.rename(columns={"text": TEXT_COLUMN})
test_clean_sinhala_df = test_clean_sinhala_df.rename(columns={"text": TEXT_COLUMN})

# ------------------------------------------------------------
# Split the ORIGINAL TRAINING SET into:
#
#   90% -> training
#   10% -> validation
#
# The existing TEST SET is NOT touched.
# ------------------------------------------------------------
train_df, valid_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df[LABEL_COLUMN]
)

# Reset dataframe indices after splitting.
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nAfter splitting:")
print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

# Check class distributions.
print("\nTrain label distribution:")
print(train_df[LABEL_COLUMN].value_counts())

print("\nValidation label distribution:")
print(valid_df[LABEL_COLUMN].value_counts())

print("\nTest label distribution:")
print(test_df[LABEL_COLUMN].value_counts())

display(train_df.head())


After splitting:
Train: (6750, 11)
Validation: (750, 11)
Test: (2500, 11)

Train label distribution:
label
0    3892
1    2858
Name: count, dtype: int64

Validation label distribution:
label
0    432
1    318
Name: count, dtype: int64

Test label distribution:
label
0    1485
1    1015
Name: count, dtype: int64


,source_id,split,condition,severity,original_text,noisy_text,label,realised_noise_rate,generator_version,n_substitutions,target_cmi
0,798491968795975680,train,misspell,heavy,@USER @USER පිංගොනා අහනෝ තෝ වැස්සි ඔන්ලයින් ඩ...,@USER @USER පිංගොනා අහනෝ තෝ වැස්සි ඔන්ලයින් ද...,1,0.090909,misspell-v2+fff1e157.495e2373,0,0
1,1216671615318216704,train,misspell,heavy,@USER අනේ උඹේ අම්මට හුකාපං පොන්න වේසිගෙ පුතා. ...,@USER අනේ උඹේ අම්මට හුකාපං පොන්න වේසියෙ පුතා. ...,1,0.052632,misspell-v2+fff1e157.495e2373,0,0
2,865966167843684352,train,misspell,heavy,@USER තෝ කොළඹ යනවා කිව්වා නේද හුත්තෝ,@USER තෝ කොකඹ යනවා කිව්වා නේද හුත්තෝ,1,0.142857,misspell-v2+fff1e157.495e2373,0,0
3,793486893417725952,train,misspell,heavy,@USER @USER තිබ්බත් සල්ලි වියදම් කරන්ඩ එපා...,@USER @USER තිබ්බත් සල්ලි වියාදම් කරන්ඩ එප...,0,0.100000,misspell-v2+fff1e157.495e2373,0,0
4,1447800275356454912,train,misspell,heavy,"රටේ දරුවො ගැන නොහිතන කාලකණ්ණි පාලකයෝ., පටු වාස...","රටේ දරුවො ගැන නොහිතන කාලකණ්ණි පාලකයෝ., පටු වසි...",1,0.177778,misspell-v2+fff1e157.495e2373,0,0


In [5]:
# ============================================================
# 05. Inspect dataset
# ============================================================

print("Columns:")
print(train_df.columns.tolist())

print("\nFirst few samples:")
display(train_df.head())

print("\nLabel distribution:")
print(train_df[LABEL_COLUMN].value_counts())

print("\nLabel distribution (%):")
print(train_df[LABEL_COLUMN].value_counts(normalize=True) * 100)

Columns:
['source_id', 'split', 'condition', 'severity', 'original_text', 'noisy_text', 'label', 'realised_noise_rate', 'generator_version', 'n_substitutions', 'target_cmi']

First few samples:


,source_id,split,condition,severity,original_text,noisy_text,label,realised_noise_rate,generator_version,n_substitutions,target_cmi
0,798491968795975680,train,misspell,heavy,@USER @USER පිංගොනා අහනෝ තෝ වැස්සි ඔන්ලයින් ඩ...,@USER @USER පිංගොනා අහනෝ තෝ වැස්සි ඔන්ලයින් ද...,1,0.090909,misspell-v2+fff1e157.495e2373,0,0
1,1216671615318216704,train,misspell,heavy,@USER අනේ උඹේ අම්මට හුකාපං පොන්න වේසිගෙ පුතා. ...,@USER අනේ උඹේ අම්මට හුකාපං පොන්න වේසියෙ පුතා. ...,1,0.052632,misspell-v2+fff1e157.495e2373,0,0
2,865966167843684352,train,misspell,heavy,@USER තෝ කොළඹ යනවා කිව්වා නේද හුත්තෝ,@USER තෝ කොකඹ යනවා කිව්වා නේද හුත්තෝ,1,0.142857,misspell-v2+fff1e157.495e2373,0,0
3,793486893417725952,train,misspell,heavy,@USER @USER තිබ්බත් සල්ලි වියදම් කරන්ඩ එපා...,@USER @USER තිබ්බත් සල්ලි වියාදම් කරන්ඩ එප...,0,0.100000,misspell-v2+fff1e157.495e2373,0,0
4,1447800275356454912,train,misspell,heavy,"රටේ දරුවො ගැන නොහිතන කාලකණ්ණි පාලකයෝ., පටු වාස...","රටේ දරුවො ගැන නොහිතන කාලකණ්ණි පාලකයෝ., පටු වසි...",1,0.177778,misspell-v2+fff1e157.495e2373,0,0



Label distribution:
label
0    3892
1    2858
Name: count, dtype: int64

Label distribution (%):
label
0    57.659259
1    42.340741
Name: proportion, dtype: float64


In [6]:
# ============================================================
# 07. Keep only text and label columns
# ============================================================

train_df = train_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
valid_df = valid_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_df = test_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()

test_transliteration_df = test_transliteration_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_codemix_df = test_codemix_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_clean_sinhala_df = test_clean_sinhala_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_misspell_df = test_misspell_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_compound_df = test_compound_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()

# Remove missing values.
train_df = train_df.dropna()
valid_df = valid_df.dropna()
test_df = test_df.dropna()

test_transliteration_df = test_transliteration_df.dropna()
test_codemix_df = test_codemix_df.dropna()
test_clean_sinhala_df = test_clean_sinhala_df.dropna()
test_misspell_df = test_misspell_df.dropna()
test_compound_df = test_compound_df.dropna()

# Make sure text is a string.
train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].astype(str)
valid_df[TEXT_COLUMN] = valid_df[TEXT_COLUMN].astype(str)
test_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].astype(str)

test_transliteration_df[TEXT_COLUMN] = test_transliteration_df[TEXT_COLUMN].astype(str)
test_codemix_df[TEXT_COLUMN] = test_codemix_df[TEXT_COLUMN].astype(str)
test_clean_sinhala_df[TEXT_COLUMN] = test_clean_sinhala_df[TEXT_COLUMN].astype(str)
test_misspell_df[TEXT_COLUMN] = test_misspell_df[TEXT_COLUMN].astype(str)
test_compound_df[TEXT_COLUMN] = test_compound_df[TEXT_COLUMN].astype(str)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

Train: (6750, 2)
Validation: (750, 2)
Test: (2500, 2)


In [7]:
# ============================================================
# 08. Encode labels
# ============================================================

# Convert NumPy integer labels to normal Python integers.
train_df[LABEL_COLUMN] = train_df[LABEL_COLUMN].astype(int)
valid_df[LABEL_COLUMN] = valid_df[LABEL_COLUMN].astype(int)
test_df[LABEL_COLUMN] = test_df[LABEL_COLUMN].astype(int)

test_transliteration_df[LABEL_COLUMN] = test_transliteration_df[LABEL_COLUMN].astype(int)
test_codemix_df[LABEL_COLUMN] = test_codemix_df[LABEL_COLUMN].astype(int)
test_clean_sinhala_df[LABEL_COLUMN] = test_clean_sinhala_df[LABEL_COLUMN].astype(int)
test_misspell_df[LABEL_COLUMN] = test_misspell_df[LABEL_COLUMN].astype(int)
test_compound_df[LABEL_COLUMN] = test_compound_df[LABEL_COLUMN].astype(int)

# Define the mapping explicitly.
label2id = {
    "non-hate": 0,
    "hate": 1,
}

id2label = {
    0: "non-hate",
    1: "hate",
}

# Create the numeric "labels" column used by the model.
train_df["labels"] = train_df[LABEL_COLUMN]
valid_df["labels"] = valid_df[LABEL_COLUMN]
test_df["labels"] = test_df[LABEL_COLUMN]

test_transliteration_df["labels"] = test_transliteration_df[LABEL_COLUMN]
test_codemix_df["labels"] = test_codemix_df[LABEL_COLUMN]
test_clean_sinhala_df["labels"] = test_clean_sinhala_df[LABEL_COLUMN]
test_misspell_df["labels"] = test_misspell_df[LABEL_COLUMN]
test_compound_df["labels"] = test_compound_df[LABEL_COLUMN]

print("label2id:", label2id)
print("id2label:", id2label)

label2id: {'non-hate': 0, 'hate': 1}
id2label: {0: 'non-hate', 1: 'hate'}


In [8]:
# ============================================================
# 09. Convert pandas DataFrames to Hugging Face Dataset
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

valid_dataset = Dataset.from_pandas(
    valid_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset,
    "test": test_dataset,
})

print(dataset)

test_transliteration_dataset = Dataset.from_pandas(
    test_transliteration_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)
test_codemix_dataset = Dataset.from_pandas(
    test_codemix_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)
test_clean_sinhala_dataset = Dataset.from_pandas(
    test_clean_sinhala_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)
test_misspell_dataset = Dataset.from_pandas(
    test_misspell_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)
test_compound_dataset = Dataset.from_pandas(
    test_compound_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

other_test_dataset = DatasetDict({
    "transliteration_test": test_transliteration_dataset,
    "codemix_test": test_codemix_dataset,
    "clean_sinhala_test": test_clean_sinhala_dataset,
    "misspell_test": test_misspell_dataset,
    "compound_test": test_compound_dataset
})

DatasetDict({
    train: Dataset({
        features: ['noisy_text', 'labels'],
        num_rows: 6750
    })
    validation: Dataset({
        features: ['noisy_text', 'labels'],
        num_rows: 750
    })
    test: Dataset({
        features: ['noisy_text', 'labels'],
        num_rows: 2500
    })
})


In [9]:
# ============================================================
# 10. Load original xlmt tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Tokenizer loaded.
Vocabulary size: 52000


In [10]:
# ============================================================
# 11. Inspect tokenization
# ============================================================

sample_texts = train_df[TEXT_COLUMN].head(10).tolist()

for text in sample_texts:

    tokens = tokenizer.tokenize(text)

    print("=" * 80)
    print("TEXT:")
    print(text)

    print("\nTOKENS:")
    print(tokens)

    print("\nNUMBER OF TOKENS:")
    print(len(tokens))

TEXT:
@USER  @USER පිංගොනා අහනෝ තෝ වැස්සි ඔන්ලයින් දැ කියල යාලුවට කියන්න..

TOKENS:
['@', 'U', 'S', 'E', 'R', 'Ġ', 'Ġ', '@', 'U', 'S', 'E', 'R', 'Ġà¶´', 'à·Ĵà¶Ĥ', 'à¶ľ', 'à·ľ', 'à¶±', 'à·ı', 'Ġà¶ħà·Ħà¶±', 'à·Ŀ', 'Ġà¶Ń', 'à·Ŀ', 'Ġà·Ģ', 'à·Ĳ', 'à·ĥ', 'à·Ĭ', 'à·ĥ', 'à·Ĵ', 'Ġà¶Ķà¶±', 'à·Ĭ', 'à¶½à¶º', 'à·Ĵ', 'à¶±', 'à·Ĭ', 'Ġà¶¯', 'à·Ĳ', 'Ġà¶ļ', 'à·Ĵ', 'à¶ºà¶½', 'Ġà¶º', 'à·ı', 'à¶½', 'à·Ķ', 'à·Ģà¶§', 'Ġà¶ļ', 'à·Ĵ', 'à¶ºà¶±', 'à·Ĭ', 'à¶±', '.', '.']

NUMBER OF TOKENS:
51
TEXT:
@USER අනේ උඹේ අම්මට හුකාපං පොන්න වේසියෙ පුතා. යස්මින් සූකා කියන්නෙ කව්ද කියලා තෝ අපිට කියන්න එපා පකෝ. ගොන් කැරියො තොපි මොන මඩ ප්‍රචාර ඇරියත් තොපිට ආයෙත් මේ රට කාබාසිනියා කරන්න ඩේන්නෙ නෑ පකෝ. සක්කිලි වහල් වේසිගේ පුතා

TOKENS:
['@', 'U', 'S', 'E', 'R', 'Ġà¶ħà¶±', 'à·ļ', 'Ġà¶ĭà¶¹', 'à·ļ', 'Ġà¶ħà¶¸', 'à·Ĭ', 'à¶¸à¶§', 'Ġà·Ħ', 'à·Ķ', 'à¶ļ', 'à·ı', 'à¶´', 'à¶Ĥ', 'Ġà¶´', 'à·ľ', 'à¶±', 'à·Ĭ', 'à¶±', 'Ġà·Ģ', 'à·ļ', 'à·ĥ', 'à·Ĵ', 'à¶º', 'à·Ļ', 'Ġà¶´', 'à·Ķ', 'à¶Ń', 'à·ı.', 'Ġà¶ºà·ĥ', 'à·Ĭ', 'à¶¸', 'à·Ĵ', 'à¶±', 'à·Ĭ', 'Ġà·ĥ', 'à·ĸ

In [11]:
# ============================================================
# 12. Tokenize dataset
# ============================================================

def tokenize_function(examples):
    """
    Tokenize the input text using the original xlmt tokenizer.

    truncation=True:
        Truncate examples longer than MAX_LENGTH.

    max_length=MAX_LENGTH:
        Maximum sequence length.

    We intentionally do not perform additional text cleaning here.
    """

    return tokenizer(
        examples[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
)

tokenized_test_dataset = other_test_dataset.map(
    tokenize_function,
    batched=True,
)

print(tokenized_dataset)
print(tokenized_test_dataset)

Map:   0%|          | 0/6750 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['noisy_text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 6750
    })
    validation: Dataset({
        features: ['noisy_text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 750
    })
    test: Dataset({
        features: ['noisy_text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2500
    })
})
DatasetDict({
    transliteration_test: Dataset({
        features: ['noisy_text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2500
    })
    codemix_test: Dataset({
        features: ['noisy_text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2500
    })
    clean_sinhala_test: Dataset({
        features: ['noisy_text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2500
    })
    misspell_test: Dataset({
        features: ['noisy_text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2500
    })
    compound_test: Dataset({
        feat

In [12]:
# ============================================================
# 13. Data collator
# ============================================================

# Instead of padding every example to 128 tokens during preprocessing, use dynamic padding.
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator created.")

Data collator created.


In [13]:
# ============================================================
# 14. Load sequence classification model
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

print("Model loaded.")

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: NLPC-UOM/SinBERT-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

In [14]:
# ============================================================
# 15. FULL model FINE-TUNING
# ============================================================

# Do NOT freeze any parameters.

for param in model.parameters():
    param.requires_grad = True

# Verify
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.2f}%")

Total parameters:     125,979,650
Trainable parameters: 125,979,650
Trainable percentage: 100.00%


In [15]:
# ============================================================
# 16. Verify which parameters are trainable
# ============================================================

total_params = 0
trainable_params = 0

for name, param in model.named_parameters():

    total_params += param.numel()

    if param.requires_grad:
        trainable_params += param.numel()

        print("TRAINABLE:", name)


print("\n" + "=" * 60)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.4f}%"
)

TRAINABLE: roberta.embeddings.word_embeddings.weight
TRAINABLE: roberta.embeddings.token_type_embeddings.weight
TRAINABLE: roberta.embeddings.LayerNorm.weight
TRAINABLE: roberta.embeddings.LayerNorm.bias
TRAINABLE: roberta.embeddings.position_embeddings.weight
TRAINABLE: roberta.encoder.layer.0.attention.self.query.weight
TRAINABLE: roberta.encoder.layer.0.attention.self.query.bias
TRAINABLE: roberta.encoder.layer.0.attention.self.key.weight
TRAINABLE: roberta.encoder.layer.0.attention.self.key.bias
TRAINABLE: roberta.encoder.layer.0.attention.self.value.weight
TRAINABLE: roberta.encoder.layer.0.attention.self.value.bias
TRAINABLE: roberta.encoder.layer.0.attention.output.dense.weight
TRAINABLE: roberta.encoder.layer.0.attention.output.dense.bias
TRAINABLE: roberta.encoder.layer.0.attention.output.LayerNorm.weight
TRAINABLE: roberta.encoder.layer.0.attention.output.LayerNorm.bias
TRAINABLE: roberta.encoder.layer.0.intermediate.dense.weight
TRAINABLE: roberta.encoder.layer.0.intermediat

In [16]:
# ============================================================
# 17. Evaluation metrics
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    # Convert logits into predicted class IDs.
    predictions = np.argmax(logits, axis=-1)

    # Accuracy.
    accuracy = accuracy_score(
        labels,
        predictions
    )

    # Macro precision, recall and F1.
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    # Weighted F1.
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

In [17]:
# ============================================================
# 18. Training configuration
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training.
    num_train_epochs=5,

    # Batch sizes.
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # a lower learning rate.
    # For large models, you may reduce this e.g. 1e-5
    learning_rate=2e-5,

    # Regularization.
    weight_decay=0.01,

    # For large model set up warmup ratio
    # warmup_ratio=0.06,
    
    # Evaluation.
    eval_strategy="epoch",

    # Save the model at each epoch.
    save_strategy="epoch",

    # Keep the best checkpoint.
    load_best_model_at_end=True,

    # Select the best model using Macro F1.
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # Logging.
    logging_strategy="steps",
    logging_steps=50,

    # Use GPU if available.
    fp16=torch.cuda.is_available(),

    # Reproducibility.
    seed=SEED,

    # Reduce unnecessary checkpoint storage.
    save_total_limit=2,

    report_to="none",
)

In [18]:
# ============================================================
# 19. Create Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer created.")

Trainer created.


In [19]:
# ============================================================
# 20. Train Model
# ============================================================

train_result = trainer.train()

print("\nTraining completed.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,0.958901,0.848686,0.796000,0.800068,0.779765,0.785088,0.792449
2,0.713298,0.870771,0.804000,0.821291,0.781730,0.789156,0.797660
3,0.573785,0.906878,0.816000,0.818107,0.802935,0.807679,0.813760
4,0.465011,1.004949,0.804000,0.801813,0.794178,0.797028,0.802746
5,0.301478,1.113351,0.809333,0.807751,0.799223,0.802348,0.807996


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (3 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Training completed.


In [20]:
# ============================================================
# 21. Evaluate on validation set
# ============================================================

validation_results = trainer.evaluate(
    tokenized_dataset["validation"]
)

print("\nValidation results:")

for key, value in validation_results.items():
    print(f"{key}: {value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Validation results:
eval_loss: 0.9078269600868225
eval_accuracy: 0.816
eval_macro_precision: 0.8181067694001101
eval_macro_recall: 0.8029350104821802
eval_macro_f1: 0.8076794434410328
eval_weighted_f1: 0.8137598501572011
eval_runtime: 3.1858
eval_samples_per_second: 235.418
eval_steps_per_second: 3.767
epoch: 5.0


In [21]:
# ============================================================
# 22. Final test evaluation
# ============================================================

test_results = trainer.evaluate(
    tokenized_dataset["test"]
)

print("\nTest results:")

for key, value in test_results.items():
    print(f"{key}: {value}")


#### Transliteration Set Test

test_results_on_transliteration = trainer.evaluate(
    tokenized_test_dataset["transliteration_test"]
)
print("\nTest on Transliteration")
for key, value in test_results_on_transliteration.items():
    print(f"{key}: {value}")


#### Codemix Set Test

test_results_on_codemix = trainer.evaluate(
    tokenized_test_dataset["codemix_test"]
)
print("\nTest on Codemix")
for key, value in test_results_on_codemix.items():
    print(f"{key}: {value}")


#### Clean Sinhala Set Test

test_results_on_clean = trainer.evaluate(
    tokenized_test_dataset["clean_sinhala_test"]
)
print("\nTest on Clean Sinhala")
for key, value in test_results_on_clean.items():
    print(f"{key}: {value}")


#### Misspell Set Test

test_results_on_misspell = trainer.evaluate(
    tokenized_test_dataset["misspell_test"]
)
print("\nTest on Misspell")
for key, value in test_results_on_misspell.items():
    print(f"{key}: {value}")


#### Compound Set Test

test_results_on_compound = trainer.evaluate(
    tokenized_test_dataset["compound_test"]
)
print("\nTest on Compound")
for key, value in test_results_on_compound.items():
    print(f"{key}: {value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Test results:
eval_loss: 0.9388704895973206
eval_accuracy: 0.8068
eval_macro_precision: 0.8043402777777777
eval_macro_recall: 0.7907564976530495
eval_macro_f1: 0.7956083297011143
eval_weighted_f1: 0.8045999280609029
eval_runtime: 10.4315
eval_samples_per_second: 239.659
eval_steps_per_second: 3.835
epoch: 5.0


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Test on Transliteration
eval_loss: 1.9636759757995605
eval_accuracy: 0.5936
eval_macro_precision: 0.2969187675070028
eval_macro_recall: 0.49966329966329964
eval_macro_f1: 0.37248995983935745
eval_weighted_f1: 0.4425180722891567
eval_runtime: 10.6046
eval_samples_per_second: 235.747
eval_steps_per_second: 3.772
epoch: 5.0


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Test on Codemix
eval_loss: 0.9432410001754761
eval_accuracy: 0.8048
eval_macro_precision: 0.8021050892656967
eval_macro_recall: 0.7887611749680715
eval_macro_f1: 0.793533328459366
eval_weighted_f1: 0.8026006839800817
eval_runtime: 10.5086
eval_samples_per_second: 237.901
eval_steps_per_second: 3.806
epoch: 5.0


... (2 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Test on Misspell
eval_loss: 0.9388704895973206
eval_accuracy: 0.8068
eval_macro_precision: 0.8043402777777777
eval_macro_recall: 0.7907564976530495
eval_macro_f1: 0.7956083297011143
eval_weighted_f1: 0.8045999280609029
eval_runtime: 10.5883
eval_samples_per_second: 236.109
eval_steps_per_second: 3.778
epoch: 5.0


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Test on Compound
eval_loss: 1.277165174484253
eval_accuracy: 0.738
eval_macro_precision: 0.7541561190048451
eval_macro_recall: 0.6982319085767361
eval_macro_f1: 0.7031121641337374
eval_weighted_f1: 0.722245516359109
eval_runtime: 10.4343
eval_samples_per_second: 239.595
eval_steps_per_second: 3.834
epoch: 5.0


In [22]:
# ============================================================
# 23. Detailed classification report
# ============================================================

predictions = trainer.predict(
    tokenized_dataset["test"]
)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=-1
)

true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

    non-hate     0.8131    0.8761    0.8434      1485
        hate     0.7956    0.7054    0.7478      1015

    accuracy                         0.8068      2500
   macro avg     0.8043    0.7908    0.7956      2500
weighted avg     0.8060    0.8068    0.8046      2500



In [23]:
# ============================================================
# 24. Detailed classification report - Transliteration
# ============================================================

predictions_tr = trainer.predict(
    tokenized_test_dataset["transliteration_test"]
)

predicted_labels_tr = np.argmax(
    predictions_tr.predictions,
    axis=-1
)

true_labels_tr = predictions_tr.label_ids

print(
    classification_report(
        true_labels_tr,
        predicted_labels_tr,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

    non-hate     0.5938    0.9993    0.7450      1485
        hate     0.0000    0.0000    0.0000      1015

    accuracy                         0.5936      2500
   macro avg     0.2969    0.4997    0.3725      2500
weighted avg     0.3527    0.5936    0.4425      2500



In [24]:
# ============================================================
# 24. Detailed classification report - CodeMix
# ============================================================

predictions_cm = trainer.predict(
    tokenized_test_dataset["codemix_test"]
)

predicted_labels_cm = np.argmax(
    predictions_cm.predictions,
    axis=-1
)

true_labels_cm = predictions_cm.label_ids

print(
    classification_report(
        true_labels_cm,
        predicted_labels_cm,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

    non-hate     0.8118    0.8741    0.8418      1485
        hate     0.7925    0.7034    0.7453      1015

    accuracy                         0.8048      2500
   macro avg     0.8021    0.7888    0.7935      2500
weighted avg     0.8039    0.8048    0.8026      2500



In [25]:
# ============================================================
# 24. Detailed classification report - Clean Sinhala
# ============================================================

predictions_cs = trainer.predict(
    tokenized_test_dataset["clean_sinhala_test"]
)

predicted_labels_cs = np.argmax(
    predictions_cs.predictions,
    axis=-1
)

true_labels_cs = predictions_cs.label_ids

print(
    classification_report(
        true_labels_cs,
        predicted_labels_cs,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

    non-hate     0.8066    0.9044    0.8527      1485
        hate     0.8299    0.6828    0.7492      1015

    accuracy                         0.8144      2500
   macro avg     0.8183    0.7936    0.8009      2500
weighted avg     0.8161    0.8144    0.8107      2500



In [26]:
# ============================================================
# 24. Detailed classification report - Misspell
# ============================================================

predictions_misspell = trainer.predict(
    tokenized_test_dataset["misspell_test"]
)

predicted_labels_misspell = np.argmax(
    predictions_misspell.predictions,
    axis=-1
)

true_labels_misspell = predictions_misspell.label_ids

print(
    classification_report(
        true_labels_misspell,
        predicted_labels_misspell,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

    non-hate     0.8131    0.8761    0.8434      1485
        hate     0.7956    0.7054    0.7478      1015

    accuracy                         0.8068      2500
   macro avg     0.8043    0.7908    0.7956      2500
weighted avg     0.8060    0.8068    0.8046      2500



In [27]:
# ============================================================
# 24. Detailed classification report - Compound
# ============================================================

predictions_compound = trainer.predict(
    tokenized_test_dataset["compound_test"]
)

predicted_labels_compound = np.argmax(
    predictions_compound.predictions,
    axis=-1
)

true_labels_compound = predictions_compound.label_ids

print(
    classification_report(
        true_labels_compound,
        predicted_labels_compound,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

    non-hate     0.7217    0.9098    0.8049      1485
        hate     0.7866    0.4867    0.6013      1015

    accuracy                         0.7380      2500
   macro avg     0.7542    0.6982    0.7031      2500
weighted avg     0.7481    0.7380    0.7222      2500



In [28]:
# ============================================================
# 25: Confusion matrix
# ============================================================

cm = confusion_matrix(
    true_labels,
    predicted_labels
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1301  184]
 [ 299  716]]


In [29]:
# ============================================================
# 26: Save trained model and tokenizer
# ============================================================

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved to:")
print(FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to:
/kaggle/working/xlmt_full_baseline_final


In [30]:
# ============================================================
# 27: Save experiment results
# ============================================================

experiment_results = {
    "experiment": "Full [MODEL_NAME] Fine Tuning", # Change accordingly
    "model": MODEL_NAME,
    "tokenizer": "Original [MODEL_NAME] tokenizer", # Change accordingly
    "encoder_trainable": True,
    "classification_head_trainable": True,
    "max_length": MAX_LENGTH,
    "seed": SEED,

    "accuracy": test_results.get("eval_accuracy"),
    "macro_precision": test_results.get("eval_macro_precision"),
    "macro_recall": test_results.get("eval_macro_recall"),
    "macro_f1": test_results.get("eval_macro_f1"),
    "weighted_f1": test_results.get("eval_weighted_f1"),

    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "trainable_percentage": (
        100 * trainable_params / total_params
    ),
}

results_df = pd.DataFrame([experiment_results])

display(results_df)

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print("Results saved.")

,experiment,model,tokenizer,encoder_trainable,classification_head_trainable,max_length,seed,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,total_parameters,trainable_parameters,trainable_percentage
0,Full xlmt Fine Tuning,NLPC-UOM/SinBERT-large,Original xlmt tokenizer,True,True,128,42,0.8068,0.80434,0.790756,0.795608,0.8046,125979650,125979650,100.0


Results saved.


In [31]:
# ============================================================
# 28: Test individual examples
# ============================================================

def predict_text(text):

    # Tokenize input text.
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    # Move inputs to the same device as the model.
    device = next(model.parameters()).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Disable gradient calculation because we are predicting.
    with torch.no_grad():

        outputs = model(**inputs)

    # Convert logits to probabilities.
    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    predicted_id = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    return {
        "text": text,
        "prediction": id2label[predicted_id],
        "confidence": probabilities[0, predicted_id].item(),
    }


# Try a few samples from your test set.
for text in test_df[TEXT_COLUMN].head(10):

    result = predict_text(text)

    print("=" * 80)
    print(result)

{'text': 'තේ නෙවෙටි තෝ බීලා ඉණ්නෙ ගිනි වතුර', 'prediction': 'hate', 'confidence': 0.8749421834945679}
{'text': '@USER තුනක් ඕනේ නෑ එකක් ගැහුවනම් ඇති ලංකාවට ආපු ගමන් ඉල්ලා අස් වෙනවා කියල', 'prediction': 'non-hate', 'confidence': 0.9928491115570068}
{'text': '@USER ,  @USER  @USER ටහුඩු ඒකි තනියම ජීවත් වෙන්න වෙරදරන එක්කෙනෙක් වෙන්න ඇති', 'prediction': 'non-hate', 'confidence': 0.9870943427085876}
{'text': '@USER මර්විනුත් මුද්දරයක් වේයිද දන්නේ නා', 'prediction': 'non-hate', 'confidence': 0.9854300022125244}
{'text': '@USER ගොසිප් පීපල් නෙමේ හුත්තො. තෝ වගේ හොරෙක්ර මේලෙ ආයෙ දඩයං හොයන්න දෙණ්නෑ. එච්චරයි', 'prediction': 'hate', 'confidence': 0.9962800145149231}
{'text': 'අපිට නම් පෙන්නන්න තියෙන්නේ මේ ගලා තමා! සුදු නෑ රතුයි!   ගුඩ් මෝනිණ් ෆ්\u200dරෙන්ස්!', 'prediction': 'non-hate', 'confidence': 0.9011892080307007}
{'text': 'කේන්තියෙන්ම මං ඔයාගෙ තාත්තට මෝඩයා කියල බැන්න. තාත්තට පුදුමැයි .. එයා ඇහුව ඔයා කොහොමද මට එහෙන කතා කරේ කියල. මං කේන්තියෙන් පිපිරී පිපිරී තව හයියෙන් කෑගැහුවා. මෝඩයා බූරුවා ගොන